In [1]:
import torch 
from torch import nn
import torch.nn.functional as F
from torch import optim
from protgen.data import ProteinDataset
from protgen.models.transformer import Transformer
from protgen.models.classifier_guided import Classifier
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

Embeddings: torch.Size([32, 129, 12])
Transformer output: torch.Size([32, 129, 23])


In [12]:
#Load the datasets
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

#Create DataLoaders
train_loader = DataLoader(train_data, batch_size=512, shuffle=True)
val_loader = DataLoader(val_data, batch_size=512, shuffle=False)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

In [19]:
for x_train_batch, targets_train_batch, y ,_ in train_loader: 
    print(y)

tensor([2, 0, 1, 1, 2, 2, 2, 0, 2, 0, 1, 1, 0, 1, 2, 0, 1, 2, 0, 1, 0, 0, 2, 0,
        2, 2, 1, 1, 0, 1, 0, 2, 1, 2, 1, 1, 2, 1, 1, 0, 1, 0, 0, 2, 2, 1, 1, 0,
        2, 2, 1, 1, 2, 2, 2, 2, 0, 0, 1, 0, 0, 0, 0, 0, 2, 1, 2, 2, 1, 1, 1, 0,
        0, 1, 1, 1, 0, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0, 0, 2, 2, 1, 0, 0,
        2, 0, 1, 2, 2, 1, 1, 0, 0, 2, 0, 1, 1, 0, 2, 1, 1, 2, 2, 1, 2, 2, 0, 2,
        1, 0, 0, 1, 0, 0, 2, 0, 2, 2, 0, 0, 0, 1, 2, 2, 2, 0, 2, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 2, 1, 2, 1, 2, 2, 0, 0, 0, 1, 2, 0, 0, 2, 1, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 1, 1, 0, 2, 1, 2, 0, 0, 1, 2, 0, 2, 0, 2, 0, 1, 0, 2, 0, 2,
        2, 1, 0, 2, 1, 1, 2, 0, 2, 1, 2, 0, 0, 0, 0, 0, 0, 2, 2, 0, 1, 0, 0, 0,
        2, 1, 2, 2, 1, 1, 0, 0, 1, 2, 2, 1, 2, 2, 0, 1, 2, 1, 1, 0, 0, 0, 2, 2,
        1, 2, 0, 2, 2, 0, 0, 1, 2, 1, 2, 0, 2, 2, 1, 1, 1, 1, 0, 0, 2, 1, 2, 1,
        0, 2, 2, 1, 2, 2, 0, 1, 0, 2, 2, 0, 1, 2, 2, 2, 2, 0, 1, 1, 1, 2, 2, 2,
        1, 2, 2, 0, 2, 1, 2, 0, 2, 0, 0,

In [ ]:
model = Transformer()
#Load a trained model!
#model.load_state_dict(torch.load("model.pt"))
x = torch.randint(0, 23, (32, 129))

capture = {}

def save_hidden(module, input, output):
    
    capture["hidden"] = output.detach()  # Detach the output to avoid tracking gradients

handle = model.feed_forward_network.register_forward_hook(save_hidden)

for x in train_loader:
    model.eval()  
    with torch.no_grad():
        _ = model(x)
        hidden = capture["hidden"]


#after capturing the hidden states for all batches. remove the handle
handle.remove()

In [3]:
model = Transformer()
model.parameters()
classifier = Classifier(model)
x = torch.randint(0, 23, (32, 129))
classifier(x)


Hidden size {torch.Size([32, 129, 12])}
Hidden


tensor([[-0.6481,  1.7066, -0.1214],
        [-0.2705,  1.6902,  0.4049],
        [ 0.1197,  1.3557,  0.4969],
        [-0.5270,  0.0982,  0.7121],
        [-0.4117,  1.1269,  1.8919],
        [-1.1087,  0.0585,  1.3465],
        [ 0.8263,  1.3160,  1.2873],
        [-0.5436,  1.1691, -0.7415],
        [ 0.6157,  0.7253, -0.0698],
        [-0.7712,  0.1613, -0.5446],
        [ 0.4062,  1.4363,  1.3395],
        [-1.3754,  0.7331,  0.8091],
        [ 0.8144,  0.8527, -0.9547],
        [ 0.0390,  0.6871,  0.5799],
        [-0.1529,  0.2474,  2.4159],
        [-0.5675,  0.3669,  0.9526],
        [ 0.4060, -0.1927, -0.5580],
        [-1.3227,  0.5983,  2.6937],
        [-0.1964,  1.5135,  0.5901],
        [-1.0874,  2.2996, -0.5284],
        [-0.8567,  2.8548, -0.0773],
        [-0.7743,  2.0700,  1.1615],
        [-1.3942,  2.2655,  1.2845],
        [-1.5581,  1.1230, -0.2304],
        [ 0.2724,  0.5901,  2.5309],
        [ 0.2370,  2.4978,  1.1881],
        [-0.4081,  2.9958,  2.0055],
 